# Reproducing the control-of-plasma-instabilities experiments

This notebook is the single entry point for reproducing the numerical outputs in the submitted manuscript. It keeps the repository small by separating two tasks:

1. **Default path:** regenerate every plotted result from compact, curated numerical arrays and copy the exact submitted manuscript figures into `outputs/figures/submitted_reference/`. This is the path used by the automatic execution check.
2. **Optional path:** run the Vlasov--Maxwell solver from initial conditions. The manuscript-scale grids and the nonlinear-control optimization are expensive, so they are not executed by default.

Run all cells from top to bottom. A successful run creates `outputs/figures/regenerated/` and `outputs/figures/submitted_reference/`. Set `SHOW_INLINE_FIGURES = True` in the first code cell only if you want the notebook to embed image previews while running interactively.


In [ ]:
from pathlib import Path
import json
import importlib.util
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
SHOW_INLINE_FIGURES = False  # Set True in an interactive notebook if you want image previews inline.


def maybe_display_figures(relative_paths):
    """Optionally show saved figures inline without affecting reproducibility."""
    if not SHOW_INLINE_FIGURES:
        print(
            f"Saved {len(relative_paths)} figure files. Inline display is disabled; set SHOW_INLINE_FIGURES=True to preview them here.")
        for rel in relative_paths:
            print(" -", rel)
        return
    for rel in relative_paths:
        display(Markdown(f"**{rel}**"))
        display(Image(filename=str(ROOT / rel)))


print(f"Repository root: {ROOT}")
print(
    f"Packaged data size: {sum(p.stat().st_size for p in (ROOT / 'data').rglob('*') if p.is_file()) / 1024 ** 2:.2f} MB")
print("JAX available:", importlib.util.find_spec("jax") is not None)



## 1. Manuscript experiment map

The manuscript contains two experiments:

- **Linear/analytic control:** cancellation of the unstable Weibel transverse magnetic mode.
- **Nonlinear/numerical control:** optimization of multi-harmonic initial transverse fields for the two-stream instability.

The metadata below is included in `data/paper_metadata.json` so that the notebook has a machine-readable record of the main parameters used in the paper.


In [ ]:

metadata = json.loads((ROOT / "data" / "paper_metadata.json").read_text())
print(json.dumps(metadata, indent=2))


## 2. Regenerate all notebook outputs

This cell regenerates the compact-data figures and copies the exact submitted PNGs. The command-line script `scripts/reproduce_all_figures.py` calls the same `make_all_outputs(...)` implementation, so the notebook and CLI paths share one code path.


In [ ]:
from vm_control.plotting import make_all_outputs

outputs = make_all_outputs(ROOT / "outputs" / "figures")
outputs = {
    "regenerated": sorted(outputs["regenerated"]),
    "submitted_reference": sorted(outputs["submitted_reference"]),
}
print(f"Regenerated figures: {len(outputs['regenerated'])}")
print(f"Submitted reference figures copied: {len(outputs['submitted_reference'])}")
print("Regenerated directory:", ROOT / "outputs" / "figures" / "regenerated")
print("Reference directory:   ", ROOT / "outputs" / "figures" / "submitted_reference")



## 3. Weibel linear-control experiment

The key claim is that the unstable transverse magnetic perturbation can be cancelled by choosing the exterior field so that the net initial unstable mode vanishes. The notebook regenerates:

- the dispersion landscape for the unstable mode,
- equilibrium and field-energy histories,
- Fourier-mode histories,
- the kinetic-energy objective landscapes,
- sensitivity to small mismatch in the cancellation amplitude.


In [ ]:
from vm_control.data import load_npz

disp = load_npz("weibel_dispersion_grid.npz")
minimum = {
    "abs_D2": float(disp["minimum_abs_D2"]),
    "sigma": float(disp["minimum_sigma"]),
    "omega": float(disp["minimum_omega"]),
}
print("Grid-search minimum for |D2| with k=1.25 from the packaged dispersion grid:")
print(minimum)

z = load_npz("weibel_fig4_histories.npz")
summary = {
    "no_control_max_B_energy": float(np.max(z["no_B"])),
    "control_max_B_energy": float(np.max(z["ctrl_B"])),
    "no_control_final_delta_K": float(z["no_K"][-1] - z["no_K"][0]),
    "control_final_delta_K": float(z["ctrl_K"][-1] - z["ctrl_K"][0]),
}
print(json.dumps(summary, indent=2))


In [ ]:
maybe_display_figures([
    "outputs/figures/regenerated/Figure-1-D2-landscape.regenerated.png",
    "outputs/figures/regenerated/Figure-4a-Fourier-no-control.regenerated.png",
    "outputs/figures/regenerated/Figure-4b-Fourier-with-control.regenerated.png",
    "outputs/figures/regenerated/Figure-4c-change-of-kinetic-energy.regenerated.png",
])


In [ ]:
# Sensitivity to imperfect control cancellation.
z = load_npz("weibel_deviation_histories.npz")
print("delta, max |B_hat_3|, max transverse field energy")
for i, delta in enumerate(z["deltas"]):
    max_bhat = float(np.max(np.abs(z["F_B"][i])))
    max_transverse_energy = float(np.max(z["E_y"][i] + z["B"][i]))
    print(f"{delta:8.1e}  {max_bhat:12.6e}  {max_transverse_energy:12.6e}")

maybe_display_figures([
    "outputs/figures/regenerated/Figure-6-Landscape-t-100.regenerated.png",
    "outputs/figures/regenerated/Figure-6-Landscape-t-200.regenerated.png",
    "outputs/figures/regenerated/Figure-6-Landscape-t-300.regenerated.png",
    "outputs/figures/regenerated/Figure-7-field-energy-vs-deviation.regenerated.png",
    "outputs/figures/regenerated/Figure-7-Fourier-vs-deviation.regenerated.png",
])



## 4. Two-stream nonlinear-control experiment

For the two-stream problem, the control field is parameterized by five harmonics. The shipped checkpoint `data/precomputed/two_stream_optimization.npz` stores the optimizer history and the best coefficients used in the submitted table.


In [ ]:
from vm_control.profiles import two_stream_best_params

z = load_npz("two_stream_optimization.npz")
params = two_stream_best_params()
print(f"Best objective: {float(z['best_objective']):.8e}")
print(f"Best checkpoint epoch: {int(z['epoch'])}")
print("\nBest parameters, rows k=1..5 and columns a_k,b_k,c_k,d_k:")
print(np.array2string(params, precision=8, suppress_small=False))

maybe_display_figures([
    "outputs/figures/regenerated/Figure-8a-equilibrium.regenerated.png",
    "outputs/figures/regenerated/Figure-10-loglog-loss.regenerated.png",
    "outputs/figures/regenerated/Table-1-two-stream-best-params.regenerated.png",
    "outputs/figures/regenerated/fig_landscape_a_d.regenerated.png",
    "outputs/figures/regenerated/fig_landscape_b_c.regenerated.png",
    "outputs/figures/regenerated/fig_landscape_min_cd.regenerated.png",
])



## 5. Exact submitted manuscript figures

The default run also copies every submitted PNG into `outputs/figures/submitted_reference/`. These files are the final artwork used by the manuscript, including panels that are expensive to regenerate from raw trajectories.


In [ ]:
reference_dir = ROOT / "outputs" / "figures" / "submitted_reference"
reference_figures = sorted(reference_dir.glob("*.png"))
print(f"Reference figures copied: {len(reference_figures)}")
print("First 10 files:")
for p in reference_figures[:10]:
    print(" -", p.name)

maybe_display_figures([
    "outputs/figures/submitted_reference/Figure-8c-energy-evolution.png",
    "outputs/figures/submitted_reference/Figure-11c-energy-evolution.png",
    "outputs/figures/submitted_reference/Figure-12-final-density.png",
    "outputs/figures/submitted_reference/Figure-13-various-Fourier-modes-suppression.png",
])


## 6. Optional full-solver smoke test

The default cells above do not execute the JAX Vlasov--Maxwell solver; they reproduce the paper figures from compact numerical arrays. The cell below is deliberately off by default. Setting `RUN_SOLVER_SMOKE_TEST = True` runs a very small Vlasov--Maxwell simulation through the same `solver_jit` code path. For manuscript-scale reruns, use the parameter values documented in the README and increase the grid/time settings; expect substantial runtime and memory use.


In [ ]:

RUN_SOLVER_SMOKE_TEST = False

if RUN_SOLVER_SMOKE_TEST:
    from vm_control.experiments import run_weibel_simulation, run_two_stream_simulation

    weibel_demo = run_weibel_simulation(n_x=8, n_vx=8, n_vy=8, t_end=0.1, delta_t=0.1)
    two_stream_demo = run_two_stream_simulation(n_x=8, n_vx=8, n_vy=8, t_end=0.1, delta_t=0.1, use_best_params=True)
    print("Weibel demo final kinetic energy:", float(weibel_demo["kinetic_energy"][-1]))
    print("Two-stream demo final kinetic energy:", float(two_stream_demo["kinetic_energy"][-1]))
else:
    print("Skipped. Set RUN_SOLVER_SMOKE_TEST=True to exercise solver_jit on a tiny grid.")



## 7. Completion check

The notebook is complete when both counts below are nonzero and the output directories exist.


In [ ]:

regen_count = len(list((ROOT / "outputs" / "figures" / "regenerated").glob("*.png")))
ref_count = len(list((ROOT / "outputs" / "figures" / "submitted_reference").glob("*.png")))
assert regen_count >= 19, regen_count
assert ref_count >= 41, ref_count
print(f"OK: {regen_count} regenerated figures and {ref_count} submitted reference figures are available.")
